# 07 — M1 Gate Decision

This notebook is the only authority for closing M1. It consumes persisted evidence from notebooks `00`–`06`; it does not infer success from notebook execution alone.

**Run order before this notebook:** `00 → 01 → 02 → 03 → 04 → 05 → 06 → 07`.


In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)


In [ ]:
from argos.common.reports import write_report
from argos.evaluation.m1_gate import evaluate_m1_gate, render_m1_gate_markdown

gate_report, decision = evaluate_m1_gate(paths.reports, PROJECT_CONFIG)

write_report(
    gate_report,
    paths.reports / "profiling" / "m1_gate_report.parquet",
)

decision_path = paths.docs / "decisions" / "M1_GATE.md"
decision_path.parent.mkdir(parents=True, exist_ok=True)
decision_path.write_text(
    render_m1_gate_markdown(gate_report, decision),
    encoding="utf-8",
)

gate_report[[
    "check_id", "category", "passed", "observed", "requirement"
]]


In [ ]:
status = "M1 ✅ CLOSED" if decision.closed else "M1 ❌ OPEN"
print(status)
print(f"Blocking checks: {decision.passed}/{decision.total}")
if decision.failed_checks:
    print("Failed:", ", ".join(decision.failed_checks))
print("Decision artifact:", decision_path)

assert decision.closed, (
    "M1 remains OPEN. Resolve the failed checks above and rerun the affected "
    "notebooks before starting Silver."
)


## What CLOSED means

Closure is scoped to the first **addition-risk V1**. It means the core contract/addition representation, target timestamps, key relationships, duplicate behavior and leakage register are sufficiently characterized to design Silver. It does **not** make `payments` or `sanctions` valid modeling inputs, and it does not waive point-in-time correctness.
